# 02 · Baseline prompt → image generation

**Milestone 1 deliverable: 5 prompt→image test runs**

Stable Diffusion v1.5 (latent diffusion: CLIP ViT-L/14 text encoder → U-Net denoiser in a 4×64×64 VAE latent space).
This notebook:
1. Generates the 5 baseline samples (DDIM, 30 steps, guidance 7.5, fixed seeds)
2. Shows that **our own text embeddings** can drive the U-Net directly via `prompt_embeds`
3. Previews Milestone 2: guidance scale and scheduler sweeps

In [ ]:
# --- Environment setup (Colab or university GPU) ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-username>/words-to-worlds.git"  # TODO: set after pushing
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True  # Colab only: keep data/ and outputs/ on Drive so they survive restarts

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        workdir = Path("/content/drive/MyDrive/words-to-worlds")
    else:
        workdir = Path("/content/words-to-worlds")
    if not workdir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(workdir)], check=True)
    os.chdir(workdir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # On the cluster, launch Jupyter from the repo root (or cd there)
    while not Path("pyproject.toml").exists() and Path.cwd() != Path("/"):
        os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(Path.cwd() / "src"))

import torch
print("cwd:", Path.cwd())
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Five baseline samples

In [ ]:
!python scripts/sample_prompts.py

In [ ]:
import json
from IPython.display import display
from PIL import Image
from t2i.config import resolve
out = resolve("outputs/milestone1")
print(json.dumps(json.loads((out / "metadata.json").read_text()), indent=2))
display(Image.open(out / "grid.png"))

## 2. Conditioning with external text embeddings
Encode prompts with `t2i.embeddings.TextEncoder` (the same CLIP model) and pass the `(B, 77, 768)` tensor straight to the U-Net. With the same seed, the image should match the plain-prompt path (up to fp16 noise).

In [ ]:
import torch, numpy as np
from t2i.config import load_config
from t2i.embeddings import TextEncoder
from t2i.generate import load_pipeline, generate, make_grid

cfg = load_config()
pipe = load_pipeline(cfg["diffusion"]["model_id"], "ddim")
enc = TextEncoder(cfg["text_encoder"]["name"], "clip")

prompt = ["A red double-decker bus parked next to a blue car."]
img_text = generate(pipe, prompt, steps=30, guidance_scale=7.5, seed=3)[0]
img_emb = generate(pipe, prompt, steps=30, guidance_scale=7.5, seed=3,
                   prompt_embeds=enc.encode(prompt)["hidden"])[0]
diff = np.abs(np.asarray(img_text, dtype=float) - np.asarray(img_emb, dtype=float)).mean()
print(f"mean abs pixel difference: {diff:.2f} / 255")
display(make_grid([img_text, img_emb], ["prompt string", "external CLIP embeddings"], cols=2))

## 3. Preview: guidance scale (classifier-free guidance)
`ε̂ = ε(x, ∅) + w · (ε(x, c) − ε(x, ∅))`. Low *w* gives diverse but off-prompt images; high *w* gives on-prompt but oversaturated ones.

In [ ]:
prompt = "Two dogs playing with a frisbee in a grassy park."
scales = [1.0, 3.0, 7.5, 12.0, 20.0]
imgs = [generate(pipe, [prompt], steps=30, guidance_scale=w, seed=1)[0] for w in scales]
display(make_grid(imgs, [f"w = {w}" for w in scales], cols=5))

## 4. Preview: schedulers / sampling steps

In [ ]:
import time
from t2i.generate import set_scheduler
prompt = "A cat sitting on a windowsill watching the rain at sunset."
rows, labels = [], []
for sch in ["ddim", "pndm", "euler_a", "dpmpp"]:
    set_scheduler(pipe, sch)
    for steps in [10, 25]:
        t0 = time.time()
        rows.append(generate(pipe, [prompt], steps=steps, guidance_scale=7.5, seed=4)[0])
        labels.append(f"{sch} · {steps} steps · {time.time()-t0:.1f}s")
display(make_grid(rows, labels, cols=4))
set_scheduler(pipe, "ddim")